In [44]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.checkpoint.sqlite import SqliteSaver

from langchain_core.messages import BaseMessage,HumanMessage,AIMessage 


In [45]:
from dotenv import load_dotenv 
load_dotenv()
import sqlite3

In [46]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [47]:
conn = sqlite3.connect(database="demo.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

In [48]:
#lets design the graph for chatting with the llm 

graph=StateGraph(MessagesState)

In [49]:
def chat_with_llm(state: MessagesState)-> MessagesState:
    
    response=llm.invoke(state['messages']) 
    
    return {'messages':[response]}

In [50]:
graph.add_node('chat',chat_with_llm)

graph.add_edge(START,'chat') 
graph.add_edge('chat',END)

In [51]:
workflow=graph.compile(checkpointer=checkpointer)

In [52]:
config={'configurable':{'thread_id':'32'}} 

In [53]:
# initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
initial_state={'messages':[HumanMessage(content='Do you know my name')]}

In [54]:
final_state=workflow.invoke(input=initial_state,config=config)

In [55]:
final_state

{'messages': [HumanMessage(content='Do you know my name', additional_kwargs={}, response_metadata={}, id='7d80a992-2025-4ea3-8138-68177e5ae2c5'),
  AIMessage(content="I don't have the ability to know your name unless you tell me. I'm a large language model, I don't have personal interactions or access to personal information about individuals. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name, I'd be happy to chat with you and use it in our conversation!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 40, 'total_tokens': 124, 'completion_time': 0.202975026, 'prompt_time': 0.000943792, 'queue_time': 0.058782788, 'total_time': 0.203918818}, 'model_name': 'Llama-3.3-70b-Versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--b6cc44ce-4063-47e8-9808-6f1f7e585e3

In [56]:
#This is simple short term memory which remains for the current execution only

In [57]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence